# Tutorial 02: Calibration-Aware Compilation

Standard quantum compilers map logical qubits to physical qubits based only on the device's **topology** (which qubits are connected). qb-compiler goes further: it uses **today's calibration data** -- per-qubit coherence times, per-gate error rates, and readout errors -- to choose the best physical qubits for your circuit.

This tutorial covers:

1. Understanding calibration data with `BackendProperties`, `QubitProperties`, and `GateProperties`
2. Loading calibration snapshots with `StaticCalibrationProvider`
3. Using `CachedCalibrationProvider` for automatic refresh
4. Running the `CalibrationMapper` pass directly
5. Comparing compilation with and without calibration data

## 1. Calibration Data Model

qb-compiler uses a vendor-neutral calibration schema. The key classes are:

- **`QubitProperties`** -- per-qubit T1, T2, readout error, frequency
- **`GateProperties`** -- per-gate error rate and duration for a specific qubit pair
- **`BackendProperties`** -- aggregates all qubit/gate properties plus the coupling map

Let's build a small calibration snapshot by hand to understand the data model.

In [ ]:
from qb_compiler.calibration.models.qubit_properties import QubitProperties
from qb_compiler.calibration.models.coupling_properties import GateProperties
from qb_compiler.calibration.models.backend_properties import BackendProperties

In [ ]:
# Define calibration for a 5-qubit device.
# Notice that qubits have different quality -- qubit 0 has the best T1/T2,
# while qubit 3 has significantly worse coherence.

qubit_props = [
    QubitProperties(qubit_id=0, t1_us=350.0, t2_us=180.0, readout_error=0.008, frequency_ghz=5.1),
    QubitProperties(qubit_id=1, t1_us=300.0, t2_us=150.0, readout_error=0.010, frequency_ghz=5.2),
    QubitProperties(qubit_id=2, t1_us=280.0, t2_us=140.0, readout_error=0.012, frequency_ghz=5.0),
    QubitProperties(qubit_id=3, t1_us=100.0, t2_us=60.0,  readout_error=0.035, frequency_ghz=5.3),
    QubitProperties(qubit_id=4, t1_us=320.0, t2_us=160.0, readout_error=0.009, frequency_ghz=5.1),
]

print("Qubit properties:")
for qp in qubit_props:
    print(
        f"  Q{qp.qubit_id}: T1={qp.t1_us:6.1f}us  T2={qp.t2_us:6.1f}us  "
        f"readout_err={qp.readout_error:.3f}"
    )

In [ ]:
# Define gate error rates for the CX gates on each edge of the coupling map.
# Notice the variation: (0,1) has 0.3% error while (2,3) has 3.5% error.

gate_props = [
    GateProperties(gate_type="cx", qubits=(0, 1), error_rate=0.003, gate_time_ns=300.0),
    GateProperties(gate_type="cx", qubits=(1, 0), error_rate=0.004, gate_time_ns=310.0),
    GateProperties(gate_type="cx", qubits=(1, 2), error_rate=0.005, gate_time_ns=320.0),
    GateProperties(gate_type="cx", qubits=(2, 1), error_rate=0.006, gate_time_ns=330.0),
    GateProperties(gate_type="cx", qubits=(2, 3), error_rate=0.035, gate_time_ns=500.0),
    GateProperties(gate_type="cx", qubits=(3, 2), error_rate=0.040, gate_time_ns=520.0),
    GateProperties(gate_type="cx", qubits=(3, 4), error_rate=0.008, gate_time_ns=350.0),
    GateProperties(gate_type="cx", qubits=(4, 3), error_rate=0.009, gate_time_ns=360.0),
    GateProperties(gate_type="cx", qubits=(0, 4), error_rate=0.004, gate_time_ns=310.0),
    GateProperties(gate_type="cx", qubits=(4, 0), error_rate=0.005, gate_time_ns=320.0),
]

print("Gate error rates:")
for gp in gate_props:
    print(
        f"  {gp.gate_type} ({gp.qubits[0]},{gp.qubits[1]}): "
        f"error={gp.error_rate:.4f}  duration={gp.gate_time_ns:.0f}ns"
    )

In [ ]:
# Assemble into a BackendProperties snapshot
coupling_map = [(0, 1), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 4), (4, 3), (0, 4), (4, 0)]

backend_props = BackendProperties(
    backend="demo_5q",
    provider="demo",
    n_qubits=5,
    basis_gates=("cx", "rz", "sx", "x", "id"),
    coupling_map=coupling_map,
    qubit_properties=qubit_props,
    gate_properties=gate_props,
    timestamp="2026-03-12T10:00:00Z",
)

print(f"Backend:     {backend_props.backend}")
print(f"Qubits:      {backend_props.n_qubits}")
print(f"Basis gates: {backend_props.basis_gates}")
print(f"Coupling:    {len(backend_props.coupling_map)} directed edges")

## 2. StaticCalibrationProvider

`StaticCalibrationProvider` wraps a `BackendProperties` snapshot and exposes it via the `CalibrationProvider` interface. This is the simplest provider -- it requires no network calls.

In [ ]:
from qb_compiler.calibration.static_provider import StaticCalibrationProvider

provider = StaticCalibrationProvider(backend_props)

print(f"Provider: {provider}")
print(f"Backend:  {provider.backend_name}")
print()

# Look up properties for a specific qubit
qp = provider.get_qubit_properties(0)
print(f"Qubit 0: T1={qp.t1_us}us, T2={qp.t2_us}us, readout_err={qp.readout_error}")

# Look up gate error for a specific edge
gp = provider.get_gate_properties("cx", (0, 1))
print(f"CX(0,1): error={gp.error_rate}, duration={gp.gate_time_ns}ns")

### Loading from JSON

In practice, you'll load calibration data from JSON files. qb-compiler ships with calibration snapshot fixtures for testing.

In [ ]:
import os

# Load a real calibration snapshot (shipped with qb-compiler tests)
fixture_dir = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "..", "..", "tests", "fixtures", "calibration_snapshots"
)

# List available snapshots
if os.path.isdir(fixture_dir):
    for f in sorted(os.listdir(fixture_dir)):
        if f.endswith(".json"):
            print(f"  {f}")
else:
    print("Fixture directory not found -- run from the project root.")

In [ ]:
# Load the IBM Torino snapshot
torino_path = os.path.join(fixture_dir, "ibm_torino_2026_02_15.json")

if os.path.exists(torino_path):
    torino_provider = StaticCalibrationProvider.from_json(torino_path)
    print(f"Loaded: {torino_provider}")
    print(f"Qubit count: {len(torino_provider.get_all_qubit_properties())}")
    print(f"Gate count:  {len(torino_provider.get_all_gate_properties())}")
else:
    print(f"Snapshot not found at {torino_path}")

## 3. CachedCalibrationProvider

`CachedCalibrationProvider` wraps any calibration source and automatically refreshes data when it goes stale. This is useful in long-running applications.

In [ ]:
from qb_compiler.calibration.cached_provider import CachedCalibrationProvider

# The factory function is called whenever the cache needs refreshing.
# In production this would fetch from the QubitBoost calibration_hub API.
# Here we just return our static provider.
def make_provider():
    return StaticCalibrationProvider(backend_props)

cached = CachedCalibrationProvider(
    provider_factory=make_provider,
    max_age_seconds=3600,       # refresh every hour
    hard_limit_hours=24.0,      # reject data older than 24h
)

print(f"Cached provider: {cached}")

# First access triggers the factory
qp = cached.get_qubit_properties(0)
print(f"Qubit 0 T1: {qp.t1_us}us (fetched transparently)")

# Manually invalidate the cache to force a refresh on next access
cached.invalidate()
qp = cached.get_qubit_properties(0)  # triggers factory again
print(f"After invalidate, Qubit 0 T1: {qp.t1_us}us")

## 4. CalibrationMapper Pass

The `CalibrationMapper` is qb-compiler's key differentiator. It maps logical qubits to physical qubits using a weighted scoring function that combines:

- **Gate error rates** -- prefer lower-error 2Q links
- **Qubit coherence** -- prefer higher T1/T2 qubits
- **Readout error** -- prefer lower-error measurement qubits

The algorithm uses VF2 subgraph isomorphism (via `rustworkx`) to enumerate candidate layouts, scores each one, and picks the best.

Let's run the mapper directly on a circuit.

In [ ]:
from qb_compiler.passes.mapping.calibration_mapper import (
    CalibrationMapper,
    CalibrationMapperConfig,
)
from qb_compiler.ir.circuit import QBCircuit as IRCircuit
from qb_compiler.ir.operations import QBGate

In [ ]:
# Build a 3-qubit GHZ circuit using the IR-level QBCircuit
ghz = IRCircuit(n_qubits=3, n_clbits=3, name="ghz_3")
ghz.add_gate(QBGate("h", (0,)))
ghz.add_gate(QBGate("cx", (0, 1)))
ghz.add_gate(QBGate("cx", (1, 2)))
ghz.add_measurement(0, 0)
ghz.add_measurement(1, 1)
ghz.add_measurement(2, 2)

print(f"Circuit:       {ghz}")
print(f"Gate count:    {ghz.gate_count}")
print(f"2Q gate count: {ghz.two_qubit_gate_count}")
print(f"Depth:         {ghz.depth}")

In [ ]:
# Create the CalibrationMapper with our backend properties
mapper = CalibrationMapper(
    calibration=backend_props,
    config=CalibrationMapperConfig(
        gate_error_weight=1.0,
        coherence_weight=0.5,
        readout_weight=0.3,
        max_candidates=100,
    ),
)

print(f"Mapper: {mapper}")
print(f"Pass name: {mapper.name}")

In [ ]:
# Run the mapper
context = {}
result = mapper.run(ghz, context)

print(f"Modified:    {result.modified}")
print(f"Layout:      {context.get('initial_layout')}")
print(f"Score:       {context.get('calibration_score', 0):.6f}")
print()
print(f"Mapped circuit: {result.circuit}")
print(f"Mapped gates:")
for op in result.circuit.gates:
    print(f"  {op}")

The layout tells you which physical qubits were chosen. The mapper should **avoid** qubit 3 (poor coherence, high readout error) and the (2,3) edge (high CX error), preferring the higher-quality qubits and links.

### Tuning the Scoring Weights

The `CalibrationMapperConfig` lets you adjust how much weight is given to each factor. This is useful when your workload has specific requirements.

In [ ]:
# Configuration biased towards readout quality (important for measurement-heavy circuits)
readout_heavy_config = CalibrationMapperConfig(
    gate_error_weight=0.3,
    coherence_weight=0.2,
    readout_weight=1.0,  # high weight on readout
    max_candidates=100,
)

mapper_ro = CalibrationMapper(calibration=backend_props, config=readout_heavy_config)
ctx_ro = {}
result_ro = mapper_ro.run(ghz, ctx_ro)

print(f"Readout-biased layout: {ctx_ro.get('initial_layout')}")
print(f"Score:                 {ctx_ro.get('calibration_score', 0):.6f}")

## 5. Building a Noise Model from Calibration

The `EmpiricalNoiseModel` translates calibration data into a noise model for fidelity estimation.

In [ ]:
from qb_compiler.noise.empirical_model import EmpiricalNoiseModel

# Build noise model from our static provider
noise = EmpiricalNoiseModel.from_calibration(provider)

print(f"Noise model: {noise}")
print()

# Compare gate errors across different edges
for q0, q1 in [(0, 1), (1, 2), (2, 3), (3, 4), (0, 4)]:
    err = noise.gate_error("cx", (q0, q1))
    print(f"  CX({q0},{q1}) error: {err:.4f}")

print()

# Readout errors
for q in range(5):
    ro = noise.readout_error(q)
    print(f"  Q{q} readout error: {ro:.4f}")

## 6. Fidelity Estimation

The `FidelityEstimator` combines gate errors, decoherence, and readout errors to predict the probability of a correct output.

In [ ]:
from qb_compiler.noise.fidelity_estimator import FidelityEstimator
from qb_compiler.noise.fidelity_estimator import QBCircuit as FECircuit

estimator = FidelityEstimator(default_gate_time_ns=40.0)

# Quick planning estimate -- no full circuit needed
# How does fidelity scale with the number of 2Q gates?
print("Quick fidelity estimates (gate errors only):")
for n_cx in [1, 5, 10, 20, 50, 100]:
    f = estimator.estimate_depth_limited_fidelity(
        n_two_qubit_gates=n_cx,
        avg_two_qubit_error=0.005,  # IBM Fez median
        n_one_qubit_gates=n_cx * 2,
        avg_one_qubit_error=0.0005,
    )
    print(f"  {n_cx:3d} CX gates -> estimated fidelity: {f:.4f}")

In [ ]:
# Full fidelity estimation with the noise model
# We use the FidelityEstimator's own minimal QBCircuit format
circ_for_fe = FECircuit(
    gates=[
        ("h", (0,)),
        ("cx", (0, 1)),
        ("cx", (1, 2)),
    ],
    n_qubits=3,
    measurements=frozenset({0, 1, 2}),
)

fidelity = estimator.estimate(circ_for_fe, noise)
print(f"GHZ-3 estimated fidelity on our demo device: {fidelity:.4f}")

## 7. Comparing With vs. Without Calibration

To see the impact of calibration-aware mapping, let's compare mapping results against a naive identity layout (logical qubit i -> physical qubit i).

In [ ]:
# Naive layout: logical qubit i -> physical qubit i
naive_layout = {0: 0, 1: 1, 2: 2}

# CalibrationMapper's layout
cal_layout = context.get('initial_layout', {})

print(f"Naive layout:       {naive_layout}")
print(f"Calibration layout: {cal_layout}")
print()

# Compare the 2Q gate errors for the edges used in a GHZ circuit
print("CX error comparison for GHZ chain (logical 0-1, 1-2):")

# Naive: uses physical edges (0,1) and (1,2)
naive_err_01 = noise.gate_error("cx", (naive_layout[0], naive_layout[1]))
naive_err_12 = noise.gate_error("cx", (naive_layout[1], naive_layout[2]))
naive_total = naive_err_01 + naive_err_12

# Calibration-aware: uses whatever edges the mapper chose
if cal_layout:
    cal_err_01 = noise.gate_error("cx", (cal_layout[0], cal_layout[1]))
    cal_err_12 = noise.gate_error("cx", (cal_layout[1], cal_layout[2]))
    cal_total = cal_err_01 + cal_err_12

    print(f"  Naive:       CX(0,1)={naive_err_01:.4f}  CX(1,2)={naive_err_12:.4f}  total={naive_total:.4f}")
    print(f"  Calibration: CX({cal_layout[0]},{cal_layout[1]})={cal_err_01:.4f}  "
          f"CX({cal_layout[1]},{cal_layout[2]})={cal_err_12:.4f}  total={cal_total:.4f}")

    if cal_total < naive_total:
        improvement = (1 - cal_total / naive_total) * 100
        print(f"  Calibration-aware mapping reduced total 2Q error by {improvement:.1f}%")
    else:
        print(f"  Naive layout happened to be optimal for this small example.")

## Summary

In this tutorial you learned:

- **Calibration data model** -- `QubitProperties`, `GateProperties`, and `BackendProperties` capture per-qubit and per-gate hardware characteristics
- **StaticCalibrationProvider** -- serves calibration from in-memory snapshots or JSON files
- **CachedCalibrationProvider** -- adds automatic refresh with configurable staleness limits
- **CalibrationMapper** -- maps logical qubits to physical qubits using a weighted scoring function over gate errors, coherence, and readout quality
- **EmpiricalNoiseModel** -- translates calibration data into a noise model for fidelity estimation
- **FidelityEstimator** -- predicts circuit output fidelity from gate errors, decoherence, and readout

The key insight: by using today's calibration data, the mapper avoids noisy qubits and high-error gates, improving circuit fidelity without changing the logical circuit.

Next up: [Tutorial 03 -- Budget-Aware Compilation](03-budget-aware.ipynb) shows how to compile within cost constraints.